# LeetCode 769 — Max Chunks To Make Sorted bằng Python

Link đề: https://leetcode.com/problems/max-chunks-to-make-sorted/ · Độ khó: **Medium** · Chủ đề: Array, Stack (ngăn xếp đơn điệu), Greedy (tham lam), Sorting

Skeleton LeetCode:

```python
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        ...
```

Ghi chú: mảng là một **hoán vị của `0 .. n-1`**; ta cắt thành các "khúc" liên tiếp, sắp xếp **riêng từng khúc**, ghép lại phải được mảng đã sắp xếp; hỏi **nhiều khúc nhất**. Đây là bài rất ngắn nhưng gom nhiều cách nhìn đẹp: tập hợp tiền tố, **max tiền tố**, **tổng tiền tố**, **ngăn xếp đơn điệu**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cắt mảng thành các khúc liên tiếp, sắp xếp riêng từng khúc, ghép lại phải **bằng mảng sắp xếp toàn cục**; tìm số khúc lớn nhất.
2. Nắm các công cụ Python cần thiết: `sorted`, `set`, `itertools.accumulate` (tích lũy với `max`), công thức tổng `i*(i+1)//2`, ngăn xếp bằng `list`, `itertools.permutations`.
3. Hiểu điều kiện cắt: có thể cắt sau vị trí `i` **khi và chỉ khi** `i + 1` phần tử đầu **chính xác là** `{0, 1, ..., i}`.
4. Đi qua 5 cách: thử mọi tập điểm cắt, so tập hợp tiền tố, max tiền tố bằng chỉ số, tổng tiền tố, ngăn xếp đơn điệu; đối chiếu bằng oracle vét cạn trên **mọi** hoán vị nhỏ.
5. Tránh các lỗi kinh điển: đếm điểm cố định `arr[i] == i`, so tiền tố với tiền tố đã sắp xếp theo **thứ tự** thay vì **tập hợp**, dùng `>=` thay vì `==`, đếm điểm cắt thay vì số khúc.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` độ dài `n`, là một **hoán vị** của các số nguyên trong `[0, n - 1]`.

Ta chia `arr` thành một số **khúc** (chunk, tức phân đoạn liên tiếp) rồi **sắp xếp từng khúc riêng**. Sau khi nối các khúc lại, kết quả phải bằng mảng đã sắp xếp. Trả về **số khúc lớn nhất** có thể chia.

**Ví dụ 1**

- Input: `arr = [4,3,2,1,0]`
- Output: `1`
- Giải thích: chia thành hai khúc trở lên không được; ví dụ `[4, 3]` và `[2, 1, 0]` cho `[3, 4, 0, 1, 2]` — chưa tăng dần.

**Ví dụ 2**

- Input: `arr = [1,0,2,3,4]`
- Output: `4`
- Giải thích: có thể chia `[1, 0]`, `[2, 3, 4]` (2 khúc), nhưng chia `[1, 0]`, `[2]`, `[3]`, `[4]` được **4** khúc — nhiều nhất.

**Ràng buộc:**

- `n == arr.length`
- `1 <= n <= 10`
- `0 <= arr[i] < n`
- Mọi phần tử của `arr` đều **khác nhau**.

In [1]:
from typing import List
from itertools import accumulate, permutations, product
import random, time

examples = [
    ([4, 3, 2, 1, 0], 1),
    ([1, 0, 2, 3, 4], 4),
]
for arr, expected in examples:
    print(f"arr={arr} n={len(arr)} kỳ vọng={expected}")

arr=[4, 3, 2, 1, 0] n=5 kỳ vọng=1
arr=[1, 0, 2, 3, 4] n=5 kỳ vọng=4


## 3. Kiến thức nền

### 3.1. Hoán vị của `0..n-1`

Một hoán vị của `0..n-1` chứa **mỗi số đúng một lần**. Mảng đã sắp xếp của nó luôn là `[0, 1, 2, ..., n-1]`, tức **phần tử ở vị trí `i` phải bằng `i`**. Đây là điều khiến bài dễ hơn bài tổng quát: ta biết trước mảng đích.

In [2]:
arr = [1, 0, 2, 4, 3]
print("sorted(arr) =", sorted(arr), "| = list(range(n))?", sorted(arr) == list(range(len(arr))))
print("điểm cố định (arr[i] == i):", [i for i, v in enumerate(arr) if v == i])

sorted(arr) = [0, 1, 2, 3, 4] | = list(range(n))? True
điểm cố định (arr[i] == i): [2]


### 3.2. Sắp xếp từng khúc rồi ghép lại

Cắt `arr` bằng các điểm cắt, sắp xếp từng khúc, nối lại và so với mảng đích. Ví dụ `[1,0,2,4,3]` cắt thành `[1,0] | [2] | [4,3]`.

In [3]:
def apply_chunks(arr, cuts):
    """cuts: danh sách các chỉ số kết thúc khúc (tính từ 0), phần tử cuối là n - 1."""
    out, start = [], 0
    for end in cuts:
        out += sorted(arr[start:end + 1])
        start = end + 1
    return out

arr = [1, 0, 2, 4, 3]
print(apply_chunks(arr, [1, 2, 4]), "| đã sắp xếp?", apply_chunks(arr, [1, 2, 4]) == sorted(arr))
print(apply_chunks(arr, [0, 4]),    "| đã sắp xếp?", apply_chunks(arr, [0, 4]) == sorted(arr))
print(apply_chunks(arr, [4]),       "| đã sắp xếp?", apply_chunks(arr, [4]) == sorted(arr))

[0, 1, 2, 3, 4] | đã sắp xếp? True
[1, 0, 2, 3, 4] | đã sắp xếp? False
[0, 1, 2, 3, 4] | đã sắp xếp? True


### 3.3. `set` và so sánh tập hợp

`set(a) == set(b)` so sánh **tập** (không quan tâm thứ tự). Với `arr[:i+1]` (tiền tố) ta hỏi "tiền tố này có đúng là tập `{0, ..., i}` không?".

In [4]:
arr = [1, 0, 2, 4, 3]
for i in range(len(arr)):
    pre = arr[:i + 1]
    print(f"i={i}: tiền tố {pre} == tập {{0..{i}}}? {set(pre) == set(range(i + 1))}   (so theo thứ tự: {pre == sorted(arr)[:i + 1]})")

i=0: tiền tố [1] == tập {0..0}? False   (so theo thứ tự: False)
i=1: tiền tố [1, 0] == tập {0..1}? True   (so theo thứ tự: False)
i=2: tiền tố [1, 0, 2] == tập {0..2}? True   (so theo thứ tự: False)
i=3: tiền tố [1, 0, 2, 4] == tập {0..3}? False   (so theo thứ tự: False)
i=4: tiền tố [1, 0, 2, 4, 3] == tập {0..4}? True   (so theo thứ tự: False)


### 3.4. `accumulate` với `max` và tổng tiền tố

`itertools.accumulate(arr, max)` cho **max tiền tố**: phần tử thứ `i` là `max(arr[0..i])`. `accumulate(arr)` (mặc định cộng) cho **tổng tiền tố**. Tổng `0 + 1 + ... + i = i * (i + 1) // 2`.

In [5]:
arr = [1, 0, 2, 4, 3]
print("max tiền tố :", list(accumulate(arr, max)))
print("tổng tiền tố:", list(accumulate(arr)))
print("0+1+..+i    :", [i * (i + 1) // 2 for i in range(len(arr))])

max tiền tố : [1, 1, 2, 4, 4]
tổng tiền tố: [1, 1, 3, 7, 10]
0+1+..+i    : [0, 1, 3, 6, 10]


### 3.5. Ngăn xếp đơn điệu (monotonic stack)

Ngăn xếp (stack) bằng `list`: `append` đẩy vào đỉnh, `pop()` lấy khỏi đỉnh, `st[-1]` xem đỉnh. **Đơn điệu** nghĩa là ta giữ ngăn xếp luôn sắp xếp (ở đây: tăng dần từ đáy lên đỉnh) bằng cách loại bỏ các phần tử vi phạm trước khi đẩy phần tử mới. Ta sẽ dùng nó ở Cách 5.

In [6]:
st = []
for x in [3, 5, 2, 8, 4]:
    while st and st[-1] > x:        # giữ tăng dần: bỏ những phần tử lớn hơn x
        st.pop()
    st.append(x)
    print(f"x={x}: stack = {st}")

x=3: stack = [3]
x=5: stack = [3, 5]
x=2: stack = [2]
x=8: stack = [2, 8]
x=4: stack = [2, 4]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Sau khi sắp xếp từng khúc rồi nối, vị trí `i` phải chứa giá trị `i`. Một khúc phủ các vị trí `l..r` được sắp xếp xong sẽ chứa **các giá trị của chính nó** ở đúng các vị trí `l..r`, tức khúc đó phải **chứa đúng tập `{l, l+1, ..., r}`**.

Vậy: điểm cắt giữa vị trí `i` và `i + 1` **hợp lệ khi và chỉ khi** các phần tử ở vị trí `0..i` là **đúng** `{0, ..., i}`.

Ví dụ `[1, 0, 2, 4, 3]`:

| `i` | tiền tố | có phải `{0..i}`? |
|---|---|---|
| 0 | `[1]` | không (cần `{0}`) |
| 1 | `[1, 0]` | **có** -> cắt |
| 2 | `[1, 0, 2]` | **có** -> cắt |
| 3 | `[1, 0, 2, 4]` | không (có 4, thiếu 3) |
| 4 | `[1, 0, 2, 4, 3]` | **có** (toàn bộ) -> cắt |

Ba điểm cắt (tính cả cuối mảng) = **3 khúc**: `[1,0]`, `[2]`, `[4,3]`. Số khúc lớn nhất chính là **số vị trí `i` mà tiền tố `0..i` là `{0..i}`**.

In [7]:
arr = [1, 0, 2, 4, 3]
cuts = [i for i in range(len(arr)) if set(arr[:i + 1]) == set(range(i + 1))]
print("các vị trí i cắt được:", cuts, "-> số khúc =", len(cuts))
print("kiểm tra: ghép lại có sắp xếp không?", apply_chunks(arr, cuts) == sorted(arr))

các vị trí i cắt được: [1, 2, 4] -> số khúc = 3
kiểm tra: ghép lại có sắp xếp không? True


## 5. Cách 1 — Brute Force: thử mọi tập điểm cắt

**Ý tưởng.** Có `n - 1` khe giữa các phần tử; mỗi khe **cắt** hoặc **không cắt**: `2^(n-1)` cách. Với mỗi cách: cắt, sắp xếp từng khúc, nối, so với `sorted(arr)`. Lấy số khúc lớn nhất trong các cách hợp lệ. Đúng theo định nghĩa của đề nên cũng dùng làm **oracle**.

In [8]:
def chunks_brute(arr: List[int]) -> int:
    n = len(arr)
    target = sorted(arr)
    best = 0
    for mask in range(1 << (n - 1)):                       # bit i = 1: cắt sau vị trí i
        cuts = [i for i in range(n - 1) if mask >> i & 1] + [n - 1]
        if apply_chunks(arr, cuts) == target:
            best = max(best, len(cuts))
    return best

for arr, expected in examples:
    got = chunks_brute(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, 3, 2, 1, 0] -> 1 OK
[1, 0, 2, 3, 4] -> 4 OK


**Độ phức tạp và nút thắt.** `O(2^n * n log n)`: với `n = 10` chỉ `512` cách nên chạy được, nhưng tăng `n` lên 30 là `5 * 10^8` cách. Nút thắt: ta thử cả những cách cắt **chắc chắn sai** (ví dụ cắt ngay sau số `3` khi `0` chưa xuất hiện). Điều kiện cắt ở mục 4 cho phép **kiểm tra từng điểm cắt độc lập**.

In [9]:
print(f"{'n':>3} {'số cách cắt 2^(n-1)':>22}")
for n in (5, 10, 20, 30, 40):
    print(f"{n:>3} {2 ** (n - 1):>22,}")

  n    số cách cắt 2^(n-1)
  5                     16
 10                    512
 20                524,288
 30            536,870,912
 40        549,755,813,888


## 6. Ý tưởng thuật toán / bước đột phá

Điểm cắt sau vị trí `i` hợp lệ **độc lập** với các điểm cắt khác (mục 4), và số khúc lớn nhất = **số điểm cắt hợp lệ** (cắt ở mọi điểm hợp lệ). Việc còn lại là kiểm tra nhanh "`arr[0..i]` có đúng là `{0..i}` không?". Có nhiều cách tương đương nhau, tận dụng việc `arr` là hoán vị:

- **So tập hợp trực tiếp** (Cách 2): `O(n)` mỗi `i`.
- **Max tiền tố** (Cách 3): `i + 1` giá trị **phân biệt** nằm trong `[0, i]` thì phải là cả tập đó; điều đó tương đương `max(arr[0..i]) == i`. Chỉ cần giữ một biến `mx`.
- **Tổng tiền tố** (Cách 4): `i + 1` giá trị phân biệt không âm có tổng nhỏ nhất là `0 + 1 + ... + i`, đạt đúng khi chúng là `{0..i}`.
- **Ngăn xếp đơn điệu** (Cách 5): mỗi phần tử của ngăn xếp là **max của một khúc**; khi gặp phần tử nhỏ hơn đỉnh, các khúc phía trên phải **gộp** lại. Cách này **không dùng** việc `arr` là hoán vị của `0..n-1` nên dùng được cho bài tổng quát (Max Chunks II).

| Cách | Ý tưởng |
|---|---|
| 2 | `set(arr[:i+1]) == set(range(i+1))` |
| 3 | `max(arr[:i+1]) == i` bằng biến `mx` |
| 4 | `sum(arr[:i+1]) == i(i+1)/2` |
| 5 | Ngăn xếp lưu max của từng khúc, gộp khi gặp số nhỏ hơn |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — So sánh tập hợp tiền tố

**Giải thích từng đoạn.**

- Với mỗi `i` từ `0` tới `n - 1`: nếu `set(arr[:i + 1]) == set(range(i + 1))` thì tăng đáp án.
- Mỗi lần dựng hai tập `O(i)`, nên tổng `O(n^2)`; với `n <= 10` nhỏ không đáng kể, nhưng là điểm yếu về lý thuyết.

In [10]:
def chunks_prefix_set(arr: List[int]) -> int:
    ans = 0
    for i in range(len(arr)):
        if set(arr[:i + 1]) == set(range(i + 1)):
            ans += 1
    return ans

for arr, expected in examples:
    assert chunks_prefix_set(arr) == expected
print("tập hợp tiền tố OK")

tập hợp tiền tố OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        ans = 0
        for i in range(len(arr)):
            if set(arr[:i + 1]) == set(range(i + 1)):
                ans += 1
        return ans

sol_set = Solution().maxChunksToSorted
print([sol_set(a) for a, _ in examples])

[1, 4]


**Độ phức tạp (Cách 2):** thời gian `O(n^2)`, bộ nhớ `O(n)`.

---

### 7.2. Cách 3 — Max tiền tố bằng chỉ số (lời giải chuẩn)

**Giải thích từng đoạn.**

- `mx` là max của các phần tử đã duyệt (`arr[0..i]`).
- Vì mọi phần tử **khác nhau** và nằm trong `[0, n-1]`, `i + 1` phần tử phân biệt có max bằng `i` thì chúng là **đúng** `{0, ..., i}` (có `i + 1` giá trị phân biệt trong đoạn `[0, i]` có `i + 1` số nguyên). Ngược lại nếu tiền tố là `{0..i}` thì max bằng `i`.
- Vậy tăng đáp án mỗi khi `mx == i`.

In [12]:
def chunks_prefix_max(arr: List[int]) -> int:
    mx = ans = 0
    for i, v in enumerate(arr):
        mx = max(mx, v)
        if mx == i:
            ans += 1
    return ans

for arr, expected in examples:
    assert chunks_prefix_max(arr) == expected
print("max tiền tố OK")

max tiền tố OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        mx = ans = 0
        for i, v in enumerate(arr):
            mx = max(mx, v)
            if i == mx:
                ans += 1
        return ans

sol_max = Solution().maxChunksToSorted
print([sol_max(a) for a, _ in examples])

[1, 4]


**Độ phức tạp (Cách 3):** thời gian `O(n)`, bộ nhớ `O(1)`. **Đây là cách nên nộp.**

---

### 7.3. Cách 4 — Tổng tiền tố

**Ý tưởng khác.** `i + 1` số nguyên **phân biệt**, **không âm** có tổng nhỏ nhất bằng `0 + 1 + ... + i = i(i+1)/2`, đạt dấu bằng khi và chỉ khi chúng là `{0, ..., i}`. Do đó tiền tố hợp lệ khi `sum(arr[:i+1]) == i(i+1)//2`.

**Giải thích từng đoạn.**

- Giữ `s` = tổng tiền tố hiện tại.
- Với mỗi `i`: `s += arr[i]`; nếu `s == i * (i + 1) // 2` thì đáp án tăng.

In [14]:
def chunks_prefix_sum(arr: List[int]) -> int:
    s = ans = 0
    for i, v in enumerate(arr):
        s += v
        if s == i * (i + 1) // 2:
            ans += 1
    return ans

for arr, expected in examples:
    assert chunks_prefix_sum(arr) == expected
print("tổng tiền tố OK")

tổng tiền tố OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        s = ans = 0
        for i, v in enumerate(arr):
            s += v
            if s == i * (i + 1) // 2:
                ans += 1
        return ans

sol_sum = Solution().maxChunksToSorted
print([sol_sum(a) for a, _ in examples])

[1, 4]


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(1)`.

---

### 7.4. Cách 5 — Ngăn xếp đơn điệu (gộp khúc)

**Ý tưởng khác hẳn.** Duy trì ngăn xếp mà **mỗi phần tử là giá trị lớn nhất của một khúc hiện hành**, tăng dần từ đáy lên đỉnh. Khi gặp số mới `x`:

- Nếu `x >= đỉnh` (hoặc ngăn xếp rỗng): `x` lập một **khúc mới** -> `push(x)`.
- Ngược lại `x` nhỏ hơn max của khúc ở đỉnh, nên khúc đó **không thể tách khỏi** `x`: phải **gộp** `x` vào khúc đó và cả những khúc phía dưới có max `> x`. Lấy `top = pop()` (max của khối gộp), rồi `pop` tiếp mọi khúc có max `> x`, cuối cùng `push(top)` (khối gộp có max chính là `top`).

Kết quả: `len(stack)` là số khúc.

Cách này không cần `arr` là hoán vị của `0..n-1`, nên là nền tảng cho bài tổng quát Max Chunks II (768).

In [16]:
def chunks_stack(arr: List[int]) -> int:
    st = []
    for x in arr:
        if not st or x >= st[-1]:
            st.append(x)                    # khúc mới
        else:
            top = st.pop()                  # max của khối sẽ được gộp
            while st and st[-1] > x:        # gộp thêm các khúc có max > x
                st.pop()
            st.append(top)
    return len(st)

for arr, expected in examples:
    assert chunks_stack(arr) == expected
print("ngăn xếp đơn điệu OK")

ngăn xếp đơn điệu OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        st = []
        for x in arr:
            if not st or x >= st[-1]:
                st.append(x)
            else:
                top = st.pop()
                while st and st[-1] > x:
                    st.pop()
                st.append(top)
        return len(st)

sol_stack = Solution().maxChunksToSorted
print([sol_stack(a) for a, _ in examples])

[1, 4]


**Độ phức tạp (Cách 5):** thời gian `O(n)` (mỗi phần tử đẩy/lấy tối đa một lần), bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Cách 3 trên `[1, 0, 2, 4, 3]`

| `i` | `arr[i]` | `mx` | `mx == i`? | đáp án |
|---|---|---|---|---|
| 0 | 1 | 1 | không | 0 |
| 1 | 0 | 1 | **có** | 1 |
| 2 | 2 | 2 | **có** | 2 |
| 3 | 4 | 4 | không | 2 |
| 4 | 3 | 4 | **có** | 3 |

Hàm `trace_max` in từng bước cùng khúc vừa đóng.

In [18]:
def trace_max(arr):
    mx = ans = start = 0
    print("arr =", arr)
    for i, v in enumerate(arr):
        mx = max(mx, v)
        note = ""
        if mx == i:
            ans += 1
            note = f"  CẮT: khúc arr[{start}..{i}] = {arr[start:i + 1]} (chứa đúng {{{start}..{i}}})"
            start = i + 1
        print(f"i={i} v={v} mx={mx} ans={ans}{note}")
    return ans

assert trace_max([1, 0, 2, 4, 3]) == 3
print()
assert trace_max([4, 3, 2, 1, 0]) == 1

arr = [1, 0, 2, 4, 3]
i=0 v=1 mx=1 ans=0
i=1 v=0 mx=1 ans=1  CẮT: khúc arr[0..1] = [1, 0] (chứa đúng {0..1})
i=2 v=2 mx=2 ans=2  CẮT: khúc arr[2..2] = [2] (chứa đúng {2..2})
i=3 v=4 mx=4 ans=2
i=4 v=3 mx=4 ans=3  CẮT: khúc arr[3..4] = [4, 3] (chứa đúng {3..4})

arr = [4, 3, 2, 1, 0]
i=0 v=4 mx=4 ans=0
i=1 v=3 mx=4 ans=0
i=2 v=2 mx=4 ans=0
i=3 v=1 mx=4 ans=0
i=4 v=0 mx=4 ans=1  CẮT: khúc arr[0..4] = [4, 3, 2, 1, 0] (chứa đúng {0..4})


### 8.2. Cách 4 (tổng tiền tố) trên `[1, 0, 2, 4, 3]`

Tổng mục tiêu `0, 1, 3, 6, 10`; tổng thật `1, 1, 3, 7, 10`. Khớp ở `i = 1, 2, 4` -> 3 khúc. Ở `i = 3`: tổng 7 ≠ 6 vì tiền tố chứa `4` thay cho `3`.

In [19]:
def trace_sum(arr):
    s = ans = 0
    print("arr =", arr)
    for i, v in enumerate(arr):
        s += v
        need = i * (i + 1) // 2
        hit = s == need
        ans += hit
        print(f"i={i} v={v} tổng={s:2} cần {need:2} -> {'khớp' if hit else 'lệch'}  ans={ans}")
    return ans

assert trace_sum([1, 0, 2, 4, 3]) == 3

arr = [1, 0, 2, 4, 3]
i=0 v=1 tổng= 1 cần  0 -> lệch  ans=0
i=1 v=0 tổng= 1 cần  1 -> khớp  ans=1
i=2 v=2 tổng= 3 cần  3 -> khớp  ans=2
i=3 v=4 tổng= 7 cần  6 -> lệch  ans=2
i=4 v=3 tổng=10 cần 10 -> khớp  ans=3


### 8.3. Cách 5 (ngăn xếp) trên `[1, 0, 2, 4, 3]` và `[4, 3, 2, 1, 0]`

Với `[1, 0, 2, 4, 3]`:

| `x` | thao tác | ngăn xếp |
|---|---|---|
| 1 | rỗng -> push | `[1]` |
| 0 | `0 < 1`: pop 1 (`top = 1`), không còn gì > 0, push `top` | `[1]` (khối `[1,0]`) |
| 2 | `2 >= 1` push | `[1, 2]` |
| 4 | push | `[1, 2, 4]` |
| 3 | `3 < 4`: pop 4, pop những `> 3` (không có), push 4 | `[1, 2, 4]` |

Độ dài 3. Với `[4, 3, 2, 1, 0]` mọi số nhỏ hơn đỉnh nên mọi khúc gộp thành một: độ dài 1.

In [20]:
def trace_stack(arr):
    st = []
    print("arr =", arr)
    for x in arr:
        if not st or x >= st[-1]:
            st.append(x)
            act = "push (khúc mới)"
        else:
            top = st.pop()
            merged = 1
            while st and st[-1] > x:
                st.pop()
                merged += 1
            st.append(top)
            act = f"gộp {merged} khúc (max khối = {top})"
        print(f"x={x}: {act:28} stack = {st}")
    return len(st)

assert trace_stack([1, 0, 2, 4, 3]) == 3
print()
assert trace_stack([4, 3, 2, 1, 0]) == 1

arr = [1, 0, 2, 4, 3]
x=1: push (khúc mới)              stack = [1]
x=0: gộp 1 khúc (max khối = 1)    stack = [1]
x=2: push (khúc mới)              stack = [1, 2]
x=4: push (khúc mới)              stack = [1, 2, 4]
x=3: gộp 1 khúc (max khối = 4)    stack = [1, 2, 4]

arr = [4, 3, 2, 1, 0]
x=4: push (khúc mới)              stack = [4]
x=3: gộp 1 khúc (max khối = 4)    stack = [4]
x=2: gộp 1 khúc (max khối = 4)    stack = [4]
x=1: gộp 1 khúc (max khối = 4)    stack = [4]
x=0: gộp 1 khúc (max khối = 4)    stack = [4]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (max tiền tố): `O(n)` thời gian, `O(1)` bộ nhớ, ba dòng logic.

```python
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        mx = ans = 0
        for i, v in enumerate(arr):
            mx = max(mx, v)
            if i == mx:
                ans += 1
        return ans
```

In [21]:
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        mx = ans = 0
        for i, v in enumerate(arr):
            mx = max(mx, v)
            if i == mx:
                ans += 1
        return ans

sol = Solution()
for arr, expected in examples:
    got = sol.maxChunksToSorted(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, 3, 2, 1, 0] -> 1 OK
[1, 0, 2, 3, 4] -> 4 OK


## 10. Vì sao đúng

**(a) Điều kiện cắt.** Giả sử ta cắt `arr` thành các khúc `[l_1..r_1], [l_2..r_2], ...` rồi sắp xếp từng khúc. Kết quả bằng `[0, 1, ..., n-1]` khi và chỉ khi mỗi khúc `[l..r]` sau khi sắp xếp là `[l, l+1, ..., r]`, tức khúc chứa **đúng tập** `{l..r}`. (Chiều "chỉ khi": các khúc sắp xếp xong nối lại tăng dần nên khúc thứ nhất chứa `r_1 + 1` số nhỏ nhất `0..r_1`, và quy nạp.)

**(b) Số khúc lớn nhất = số `i` thỏa "`arr[0..i]` = `{0..i}`".** Nếu ta cắt ở khe sau `i` thì phải có `arr[0..i] = {0..i}` (tập các khúc đầu hợp lại, theo (a)). Ngược lại, nếu có các vị trí `i_1 < i_2 < ... < i_m = n-1` với tiền tố `arr[0..i_t] = {0..i_t}`, thì cắt ở tất cả chúng: khúc `(i_{t-1}, i_t]` chứa `{0..i_t} \ {0..i_{t-1}} = {i_{t-1}+1 .. i_t}`: hợp lệ. Cắt ở **mọi** điểm hợp lệ cho nhiều khúc nhất vì tập điểm cắt bất kỳ hợp lệ là tập con của các điểm đó. Vị trí cuối `i = n - 1` luôn hợp lệ (toàn mảng là `{0..n-1}`) nên đáp án `>= 1`.

**(c) Cách 3.** `arr[0..i]` gồm `i + 1` số **phân biệt** thuộc `[0, n-1]`. Nếu max bằng `i` thì cả `i + 1` số đều thuộc `[0, i]` (đúng `i + 1` giá trị), nên bằng `{0..i}`. Ngược lại `{0..i}` có max `i`. Vậy "max tiền tố == i" tương đương "tiền tố = {0..i}".

**(d) Cách 4.** `i + 1` số nguyên phân biệt không âm: tổng `>= 0 + 1 + ... + i` (sắp chúng tăng dần, số thứ `j` (từ 0) `>= j`), đạt dấu bằng khi và chỉ khi số thứ `j` đúng bằng `j` với mọi `j`, tức tập là `{0..i}`.

**(e) Cách 5.** Bất biến: ngăn xếp tăng dần (không giảm), mỗi phần tử là **max của một khúc** và các khúc là phân hoạch **tối đa** (mịn nhất) của phần đã duyệt thành các khúc "sắp xếp độc lập được". Số mới `x >= đỉnh`: không có khúc nào cần gộp với `x`, ta lập khúc mới. Số mới `x < đỉnh`: khúc có max lớn hơn `x` chứa một phần tử đứng trước `x` nhưng lớn hơn `x`, nên `x` và khúc đó phải chung khúc (khác thì sau sắp xếp thứ tự sai); và mọi khúc nằm giữa cũng bị nuốt vào (khúc phải liên tiếp). Max của khối mới là `top` (max lớn nhất trong các khúc bị gộp).

**Kiểm chứng bằng máy:** so cả 5 cách (và `Solution` cuối) với oracle vét cạn trên **mọi hoán vị** độ dài 1..7 (tổng 5913 hoán vị) và trên nhiều hoán vị ngẫu nhiên có seed cho `n = 8, 9, 10`.

In [22]:
solutions = {
    "brute": chunks_brute,
    "set": chunks_prefix_set,
    "max": chunks_prefix_max,
    "sum": chunks_prefix_sum,
    "stack": chunks_stack,
    "sol_set": sol_set,
    "sol_max": sol_max,
    "sol_sum": sol_sum,
    "sol_stack": sol_stack,
    "final": Solution().maxChunksToSorted,
}

checked = 0
for n in range(1, 8):
    for perm in permutations(range(n)):
        arr = list(perm)
        ref = chunks_brute(arr)
        for name, fn in solutions.items():
            assert fn(arr) == ref, (name, arr, fn(arr), ref)
        checked += 1
print("mọi hoán vị độ dài 1..7:", checked, "hoán vị x", len(solutions), "cài đặt khớp oracle")

mọi hoán vị độ dài 1..7: 5913 hoán vị x 10 cài đặt khớp oracle


## 11. Time / Space Complexity

Gọi `n = len(arr)` (`<= 10`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập điểm cắt | `O(2^n * n log n)` | `O(n)` |
| 2. So tập hợp tiền tố | `O(n^2)` | `O(n)` |
| 3. Max tiền tố | `O(n)` | `O(1)` |
| 4. Tổng tiền tố | `O(n)` | `O(1)` |
| 5. Ngăn xếp đơn điệu | `O(n)` | `O(n)` |

Với `n <= 10` mọi cách đều tức thì. Để thấy khác biệt, ta đo (ngoài ràng buộc) với hoán vị `n = 3000` (bài `O(n^2)` bắt đầu chậm hơn), bỏ qua vét cạn.

In [23]:
random.seed(769)
n = 3000
big_perms = []
for _ in range(5):
    p = list(range(n))
    random.shuffle(p)
    big_perms.append(p)
# hoán vị nhiều khúc: xáo trộn trong từng khối 3 phần tử
blocks = []
for s in range(0, n, 3):
    blk = list(range(s, s + 3))
    random.shuffle(blk)
    blocks += blk
big_perms.append(blocks)
big_perms.append(list(range(n)))          # đồng nhất: n khúc

ref = None
for name in ["set", "max", "sum", "stack"]:
    t0 = time.perf_counter()
    res = [solutions[name](p) for p in big_perms]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:6} {len(big_perms)} hoán vị n={n}: {dt:9.2f} ms")
print("kết quả:", ref)

set    7 hoán vị n=3000:   1594.75 ms
max    7 hoán vị n=3000:      1.31 ms
sum    7 hoán vị n=3000:      1.93 ms
stack  7 hoán vị n=3000:      1.40 ms
kết quả: [1, 1, 1, 1, 1, 1675, 3000]


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle, và test ngẫu nhiên có seed cố định (đúng ràng buộc `n <= 10`).

In [24]:
# 12.1 ví dụ đề
for arr, expected in examples:
    for name, fn in solutions.items():
        assert fn(arr) == expected, (name, arr)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay)
edge = [
    ([0], 1),                                   # một phần tử
    ([0, 1], 2),                                # đã sắp xếp, mỗi phần tử một khúc
    ([1, 0], 1),                                # đảo hai phần tử: một khúc
    ([0, 1, 2, 3, 4], 5),                       # đã sắp xếp
    ([4, 3, 2, 1, 0], 1),                       # đảo ngược
    ([1, 0, 3, 2, 5, 4], 3),                    # từng cặp đảo
    ([2, 0, 1, 4, 3], 2),                       # [2,0,1] và [4,3]
    ([0, 2, 1, 3], 3),                          # [0] [2,1] [3]
    ([1, 2, 0, 3], 2),                          # [1,2,0] và [3]
    ([3, 0, 1, 2], 1),                          # 3 ở đầu kéo mọi thứ thành một khúc
    ([1, 0, 2, 3, 4], 4),                       # ví dụ của đề
    ([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 10),      # n = 10 đã sắp xếp
    ([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], 1),        # n = 10 đảo ngược
    ([1, 0, 3, 2, 5, 4, 7, 6, 9, 8], 5),
]
for arr, expected in edge:
    assert chunks_brute(arr) == expected, ("oracle khác kỳ vọng", arr)
    for name, fn in solutions.items():
        assert fn(arr) == expected, (name, arr, fn(arr), expected)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 phản ví dụ: các cách sai
def wrong_fixed_points(arr):       # đếm điểm cố định arr[i] == i
    return sum(1 for i, v in enumerate(arr) if v == i)

def wrong_sequence_prefix(arr):    # so tiền tố theo THỨ TỰ với tiền tố của mảng đã sắp xếp
    s = sorted(arr)
    return sum(1 for i in range(len(arr)) if arr[:i + 1] == s[:i + 1])

def wrong_ge(arr):                 # dùng mx >= i
    mx = ans = 0
    for i, v in enumerate(arr):
        mx = max(mx, v)
        if mx >= i:
            ans += 1
    return ans

def wrong_cuts(arr):               # đếm điểm cắt (n - 1 khe) thay vì số khúc
    mx = ans = 0
    for i, v in enumerate(arr[:-1]):
        mx = max(mx, v)
        if mx == i:
            ans += 1
    return ans

wrongs = [
    ("đếm điểm cố định", wrong_fixed_points, [1, 0, 2, 3, 4]),
    ("so tiền tố theo thứ tự", wrong_sequence_prefix, [1, 0, 2, 3, 4]),
    ("mx >= i", wrong_ge, [4, 3, 2, 1, 0]),
    ("đếm khe cắt (thiếu khúc cuối)", wrong_cuts, [1, 0, 2, 3, 4]),
]
for name, fn, bad in wrongs:
    truth = chunks_brute(bad)
    print(f"{name:30} {bad}: sai = {fn(bad)}, đúng = {truth}")
    assert fn(bad) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

# 12.4 ngẫu nhiên có seed, đúng ràng buộc n <= 10
rng = random.Random(2769)
for _ in range(600):
    n = rng.randint(8, 10)
    arr = list(range(n))
    rng.shuffle(arr)
    if rng.random() < 0.5:
        # tăng khả năng có nhiều khúc: sắp xếp lại theo từng khối ngẫu nhiên
        arr = list(range(n))
        i = 0
        while i < n:
            j = min(n, i + rng.randint(1, 4))
            blk = arr[i:j]
            rng.shuffle(blk)
            arr[i:j] = blk
            i = j
    ref = chunks_brute(arr)
    for name, fn in solutions.items():
        assert fn(arr) == ref, (name, arr)
print("12.4 ngẫu nhiên n = 8..10: OK")

# 12.5 tính chất: đáp án trong [1, n]; = n khi và chỉ khi đã sắp xếp; = 1 với hoán vị mà 0 ở cuối
for n in range(1, 8):
    for perm in permutations(range(n)):
        ans = chunks_prefix_max(list(perm))
        assert 1 <= ans <= n
        assert (ans == n) == (list(perm) == sorted(perm))
        if n > 1 and perm[-1] == 0:
            assert ans == 1
print("12.5 tính chất của đáp án: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca
đếm điểm cố định               [1, 0, 2, 3, 4]: sai = 3, đúng = 4
so tiền tố theo thứ tự         [1, 0, 2, 3, 4]: sai = 0, đúng = 4
mx >= i                        [4, 3, 2, 1, 0]: sai = 5, đúng = 1
đếm khe cắt (thiếu khúc cuối)  [1, 0, 2, 3, 4]: sai = 3, đúng = 4
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


12.4 ngẫu nhiên n = 8..10: OK
12.5 tính chất của đáp án: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm điểm cố định `arr[i] == i`

Nghĩ rằng "phần tử đã đúng chỗ" thì tự thành một khúc. Nhưng một cặp đảo như `[1, 0]` cũng là một khúc dù không có điểm cố định.

In [25]:
arr = [1, 0, 2, 3, 4]
print("sai (điểm cố định):", wrong_fixed_points(arr), "| đúng:", Solution().maxChunksToSorted(arr))
assert wrong_fixed_points(arr) == 3 and Solution().maxChunksToSorted(arr) == 4

sai (điểm cố định): 3 | đúng: 4


Vì sao sai: điểm cố định chỉ đếm `2, 3, 4`; khúc `[1, 0]` (sắp xếp được thành `[0, 1]`) bị bỏ sót. **Đúng:** điều kiện cắt là tiền tố **bằng tập** `{0..i}`, không phải từng phần tử đúng chỗ.

### Lỗi 2 — So tiền tố theo thứ tự thay vì theo tập

`arr[:i+1] == sorted(arr)[:i+1]` đòi tiền tố **đã** được sắp xếp. Nhưng ta được phép sắp xếp từng khúc.

In [26]:
arr = [1, 0, 2, 3, 4]
print("sai (so theo thứ tự):", wrong_sequence_prefix(arr), "| đúng:", Solution().maxChunksToSorted(arr))
assert wrong_sequence_prefix(arr) == 0

sai (so theo thứ tự): 0 | đúng: 4


Vì sao sai: tiền tố `[1, 0]` khác `[0, 1]` về thứ tự nên bị loại, và vì `1` đứng trước `0` ở đầu mảng nên **mọi** tiền tố sau đó cũng bị loại (kết quả 0, thậm chí không đếm nổi khúc cuối). **Đúng:** so sánh **tập hợp** (hoặc max/tổng tiền tố).

### Lỗi 3 — Dùng `mx >= i` thay vì `mx == i`

`mx >= i` luôn đúng (vì `mx >= arr[i]`... và các giá trị phân biệt), đếm mọi vị trí.

In [27]:
arr = [4, 3, 2, 1, 0]
print("sai (>=):", wrong_ge(arr), "| đúng:", Solution().maxChunksToSorted(arr))
assert wrong_ge(arr) == 5 and Solution().maxChunksToSorted(arr) == 1

sai (>=): 5 | đúng: 1


Vì sao sai: trong một hoán vị, max tiền tố **không bao giờ nhỏ hơn** `i` (có `i + 1` giá trị phân biệt không âm thì max `>= i`). Điều kiện cắt là **dấu bằng**. **Đúng:** `if mx == i`.

### Lỗi 4 — Đếm khe cắt thay vì số khúc

Số khúc = số điểm cắt hợp lệ **cộng cả điểm cuối** (khúc cuối kết thúc ở `n - 1`). Nếu chỉ duyệt `n - 1` khe, đáp án thiếu 1.

In [28]:
arr = [1, 0, 2, 3, 4]
print("sai (thiếu khúc cuối):", wrong_cuts(arr), "| đúng:", Solution().maxChunksToSorted(arr))
assert wrong_cuts(arr) == 3 and Solution().maxChunksToSorted(arr) == 4

sai (thiếu khúc cuối): 3 | đúng: 4


Vì sao sai: `n` phần tử có `n - 1` khe giữa chúng; `m` khe được cắt tạo `m + 1` khúc. **Đúng:** duyệt đủ `i = 0..n-1` (điểm `i = n - 1` luôn thỏa `mx == i`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia mảng thành các đoạn, sắp xếp riêng từng đoạn", "sắp xếp được" -> điều kiện cắt dạng "**mọi phần tử bên trái `<=` mọi phần tử bên phải**" (`max tiền tố <= min hậu tố`).
- **Hoán vị `0..n-1`:** `max tiền tố == i` (hoặc tổng tiền tố). Đây là trường hợp đặc biệt đơn giản nhất.
- **Mảng tổng quát (có thể trùng, giá trị bất kỳ):** dùng `max tiền tố <= min hậu tố` (hai mảng phụ) hoặc **ngăn xếp đơn điệu** (Cách 5).
- Cũng có thể nhìn bài là **gộp các chu trình của hoán vị**: mỗi khúc là hợp của các chu trình; số khúc lớn nhất là số "khối liên thông" theo các cạnh `i -> arr[i]`.
- Chú ý `n` phần tử có `n - 1` khe, nhưng điểm cuối cùng luôn thỏa điều kiện nên đáp án `>= 1`.
- Test nên có: đã sắp xếp (`n` khúc), đảo ngược (1 khúc), từng cặp đảo, `0` ở cuối, `n - 1` ở đầu.

## 15. Biến thể

1. **Max Chunks To Make Sorted II (768):** mảng bất kỳ (có số trùng, giá trị lớn). Dùng `max tiền tố <= min hậu tố` hoặc ngăn xếp (Cách 5 đúng nguyên xi).
2. **Chia thành các khúc trả về biên:** liệt kê các khúc cụ thể.
3. **Số khúc nhỏ nhất** là `1` (toàn mảng) — tầm thường; còn **đếm số cách chia hợp lệ** (mọi tập con của các điểm cắt hợp lệ): `2^(đáp án - 1)`.
4. Bài liên quan: Partition Labels (763: cùng khuôn "cắt khi chạm đường chân trời"), Valid Mountain, Sort Array By Parity.

Dưới đây là code cho biến thể 1 (hai cách: hậu tố nhỏ nhất và ngăn xếp) và biến thể 3, kiểm bằng oracle vét cạn.

In [29]:
def chunks_general_suffix_min(arr: List[int]) -> int:
    n = len(arr)
    suffix_min = arr[:]
    for i in range(n - 2, -1, -1):
        suffix_min[i] = min(suffix_min[i], suffix_min[i + 1])
    mx = ans = 0
    for i in range(n):
        mx = arr[i] if i == 0 else max(mx, arr[i])
        if i == n - 1 or mx <= suffix_min[i + 1]:
            ans += 1
    return ans

def chunks_general_oracle(arr: List[int]) -> int:
    n = len(arr)
    target = sorted(arr)
    best = 0
    for mask in range(1 << (n - 1)):
        cuts = [i for i in range(n - 1) if mask >> i & 1] + [n - 1]
        if apply_chunks(arr, cuts) == target:
            best = max(best, len(cuts))
    return best

def count_valid_splits(arr: List[int]) -> int:
    return 2 ** (chunks_prefix_max(arr) - 1)

print(chunks_general_suffix_min([5, 4, 3, 2, 1]), chunks_general_suffix_min([2, 1, 3, 4, 4]))
assert chunks_general_suffix_min([5, 4, 3, 2, 1]) == 1 and chunks_general_suffix_min([2, 1, 3, 4, 4]) == 4
rng = random.Random(768)
for _ in range(400):
    n = rng.randint(1, 9)
    arr = [rng.randint(0, 5) for _ in range(n)]
    ref = chunks_general_oracle(arr)
    assert chunks_general_suffix_min(arr) == ref == chunks_stack(arr), (arr, ref)
for perm in permutations(range(6)):
    arr = list(perm)
    ways = sum(1 for mask in range(1 << 5)
               if apply_chunks(arr, [i for i in range(5) if mask >> i & 1] + [5]) == sorted(arr))
    assert ways == count_valid_splits(arr), arr
print("Biến thể: OK")

1 4
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập cắt | Định nghĩa trực tiếp | `O(2^n n log n)` | `O(n)` | Oracle, `n` rất nhỏ |
| 2. Tập hợp tiền tố | `set(prefix) == {0..i}` | `O(n^2)` | `O(n)` | Dễ hiểu nhất, `n` nhỏ |
| 3. Max tiền tố | `max == i` | `O(n)` | `O(1)` | **Cách nộp chính**, riêng cho hoán vị `0..n-1` |
| 4. Tổng tiền tố | `sum == i(i+1)/2` | `O(n)` | `O(1)` | Tương đương Cách 3, cách nhìn khác |
| 5. Ngăn xếp đơn điệu | Gộp khúc khi gặp số nhỏ hơn | `O(n)` | `O(n)` | Mảng tổng quát (768), không cần hoán vị |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
arr = [1, 0, 2, 4, 3]
print(list(accumulate(arr, max)))
print([i for i, m in enumerate(accumulate(arr, max)) if m == i])
print(chunks_prefix_max(arr))
```

In [30]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `arr = [2, 0, 1, 4, 3]`: điền max tiền tố, danh sách các `i` cắt được và số khúc.

In [31]:
# Bài 2: điền kết quả tính tay
hand_prefix_max = None     # list 5 phần tử
hand_cut_positions = None  # list các i
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành bản **tổng tiền tố** (Cách 4).

In [32]:
def exercise_3_prefix_sum(arr: List[int]) -> int:
    s = ans = 0
    for i, v in enumerate(arr):
        # TODO 1: cộng v vào s
        # TODO 2: tổng nhỏ nhất của i + 1 số phân biệt không âm là i * (i + 1) // 2
        # TODO 3: nếu s bằng giá trị đó thì tăng ans
        pass
    return None


# assert exercise_3_prefix_sum([1, 0, 2, 4, 3]) == 3
# assert exercise_3_prefix_sum([4, 3, 2, 1, 0]) == 1
# assert exercise_3_prefix_sum([0, 1, 2]) == 3

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, hãy tự viết `exercise_4_stack(arr)` bằng ngăn xếp đơn điệu (mỗi phần tử ngăn xếp là max của một khúc).

In [33]:
def exercise_4_stack(arr: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_stack([1, 0, 2, 4, 3]) == 3
# assert exercise_4_stack([4, 3, 2, 1, 0]) == 1
# assert exercise_4_stack([0, 1, 2]) == 3

### Bài 5 — Mini challenge: Max Chunks II (mảng tổng quát) (khó)

Mảng `arr` gồm số nguyên **bất kỳ** (có thể trùng, `0 <= arr[i] <= 10^8`, `n <= 2000`), không còn là hoán vị. Chia thành nhiều khúc nhất sao cho sắp xếp riêng từng khúc rồi nối ra mảng đã sắp xếp.

```python
exercise_5_general([5, 4, 3, 2, 1])   -> 1
exercise_5_general([2, 1, 3, 4, 4])   -> 4
exercise_5_general([1, 1, 1])         -> 3
```
Gợi ý: cắt sau `i` khi `max(arr[:i+1]) <= min(arr[i+1:])`; hoặc dùng ngăn xếp.

In [34]:
def exercise_5_general(arr: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_general([5, 4, 3, 2, 1]) == 1
# assert exercise_5_general([2, 1, 3, 4, 4]) == 4
# assert exercise_5_general([1, 1, 1]) == 3

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `accumulate(arr, max)` giữ max tiền tố.
- Mức 2: max tiền tố của `[1, 0, 2, 4, 3]` là `1, 1, 2, 4, 4`.
- Mức 3: `[1, 1, 2, 4, 4]`, `[1, 2, 4]`, `3`.

**Bài 2**
- Mức 1: `[2, 0, 1, 4, 3]` -> max tiền tố `2, 2, 2, 4, 4`.
- Mức 2: so với chỉ số `0, 1, 2, 3, 4`: bằng nhau ở `i = 2` và `i = 4`.
- Mức 3: `[2, 2, 2, 4, 4]`, `[2, 4]`, `2` (khúc `[2,0,1]` và `[4,3]`).

**Bài 3**
- Mức 1: `s += v`.
- Mức 2: so `s == i * (i + 1) // 2`.
- Mức 3: `if s == i * (i + 1) // 2: ans += 1`; cuối `return ans`.

**Bài 4**
- Mức 1: `st = []`; với mỗi `x`: nếu `not st or x >= st[-1]` thì `push`.
- Mức 2: ngược lại `top = st.pop()`, rồi `while st and st[-1] > x: st.pop()`.
- Mức 3: `st.append(top)`; cuối `return len(st)`.

**Bài 5**
- Mức 1: cắt sau `i` khi mọi phần tử bên trái `<=` mọi phần tử bên phải.
- Mức 2: dựng `suffix_min[i]`; duyệt với `mx`.
- Mức 3: `if i == n - 1 or mx <= suffix_min[i + 1]: ans += 1`. (Hoặc dùng ngăn xếp đã viết ở Bài 4: nó đã đúng cho mảng tổng quát.)

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [35]:
arr = [1, 0, 2, 4, 3]
print(list(accumulate(arr, max)))
print([i for i, m in enumerate(accumulate(arr, max)) if m == i])
print(chunks_prefix_max(arr))

assert list(accumulate(arr, max)) == [1, 1, 2, 4, 4]
assert [i for i, m in enumerate(accumulate(arr, max)) if m == i] == [1, 2, 4]
assert chunks_prefix_max(arr) == 3

[1, 1, 2, 4, 4]
[1, 2, 4]
3


### Đáp án Bài 2

Max tiền tố `[2, 2, 2, 4, 4]`; khớp chỉ số ở `i = 2` và `i = 4` -> 2 khúc: `[2, 0, 1]` và `[4, 3]`.

In [36]:
hand_prefix_max_solution = [2, 2, 2, 4, 4]
hand_cut_positions_solution = [2, 4]
hand_answer_solution = 2
arr = [2, 0, 1, 4, 3]
assert list(accumulate(arr, max)) == hand_prefix_max_solution
assert [i for i, m in enumerate(hand_prefix_max_solution) if m == i] == hand_cut_positions_solution
assert chunks_brute(arr) == hand_answer_solution == len(hand_cut_positions_solution)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [37]:
def exercise_3_prefix_sum_solution(arr: List[int]) -> int:
    s = ans = 0
    for i, v in enumerate(arr):
        s += v
        if s == i * (i + 1) // 2:
            ans += 1
    return ans

assert exercise_3_prefix_sum_solution([1, 0, 2, 4, 3]) == 3
assert exercise_3_prefix_sum_solution([4, 3, 2, 1, 0]) == 1
assert exercise_3_prefix_sum_solution([0, 1, 2]) == 3
for perm in permutations(range(6)):
    assert exercise_3_prefix_sum_solution(list(perm)) == chunks_brute(list(perm)), perm
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [38]:
def exercise_4_stack_solution(arr: List[int]) -> int:
    st = []
    for x in arr:
        if not st or x >= st[-1]:
            st.append(x)
        else:
            top = st.pop()
            while st and st[-1] > x:
                st.pop()
            st.append(top)
    return len(st)

assert exercise_4_stack_solution([1, 0, 2, 4, 3]) == 3
assert exercise_4_stack_solution([4, 3, 2, 1, 0]) == 1
assert exercise_4_stack_solution([0, 1, 2]) == 3
for perm in permutations(range(6)):
    assert exercise_4_stack_solution(list(perm)) == chunks_brute(list(perm)), perm
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [39]:
def exercise_5_general_solution(arr: List[int]) -> int:
    n = len(arr)
    suffix_min = arr[:]
    for i in range(n - 2, -1, -1):
        suffix_min[i] = min(suffix_min[i], suffix_min[i + 1])
    mx = ans = 0
    for i in range(n):
        mx = max(mx, arr[i])
        if i == n - 1 or mx <= suffix_min[i + 1]:
            ans += 1
    return ans

assert exercise_5_general_solution([5, 4, 3, 2, 1]) == 1
assert exercise_5_general_solution([2, 1, 3, 4, 4]) == 4
assert exercise_5_general_solution([1, 1, 1]) == 3
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 9)
    arr = [rng.randint(0, 6) for _ in range(n)]
    assert exercise_5_general_solution(arr) == chunks_general_oracle(arr), arr
big = [rng.randint(0, 10 ** 8) for _ in range(2000)]
assert exercise_5_general_solution(big) == chunks_stack(big)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (max tiền tố)

```python
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        mx = ans = 0
        for i, v in enumerate(arr):
            mx = max(mx, v)        # max của arr[0..i]
            if mx == i:            # arr[0..i] đúng là {0..i} -> cắt được
                ans += 1
        return ans
```

Công thức nhớ nhanh: cắt sau `i` khi `max(arr[0..i]) == i` (hoán vị `0..n-1`); tổng quát: `max(arr[0..i]) <= min(arr[i+1..])`.

### Checklist trước khi submit

- [ ] Điều kiện là `mx == i` (dấu **bằng**), không phải `>=`?
- [ ] Duyệt đủ `i = 0..n-1` (khúc cuối được tính)?
- [ ] Đếm **số khúc**, không phải số khe cắt?
- [ ] Không dùng `arr[i] == i` (điểm cố định)?
- [ ] Không so tiền tố theo **thứ tự** với tiền tố đã sắp xếp?
- [ ] Nếu mảng không phải hoán vị `0..n-1` (bài 768): đã chuyển sang `max tiền tố <= min hậu tố` hoặc ngăn xếp?
- [ ] Đã thử: `[0]`, mảng đã sắp xếp, đảo ngược, từng cặp đảo?
- [ ] Tên method đúng `maxChunksToSorted(self, arr)` và trả `int`?